In [2]:
import pandas as pd
from langsmith import Client

from dotenv import load_dotenv
load_dotenv()

client = Client()

dataset = client.create_dataset(
    dataset_name="Telecom-RAG-dataset",
    description="Evaluation dataset for Telecom RAG Chatbot"
)

df = pd.read_csv("eval_dataset.csv")

for _, row in df.iterrows():

    client.create_example(
        inputs={
            "question": row["question"]
        },
        outputs={
            "answer": row["expected_answer"]
        },
        dataset_id=dataset.id
    )

print("Dataset uploaded successfully.")

Dataset uploaded successfully.


In [3]:
from rag_chain import build_chain

chain = build_chain()

def target(inputs):

    question = inputs["question"]

    answer = chain.invoke(question)

    return {
        "answer": answer
    }

/Users/Desilva/miniconda3/envs/ml_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-13 17:21:34.634174: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/Users/Desilva/miniconda3/envs/ml_env/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that 

In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

def semantic_match(run, example):

    actual = run.outputs["answer"]
    expected = example.outputs["answer"]

    embeddings = model.encode(
        [actual, expected]
    )

    a = embeddings[0]
    b = embeddings[1]

    similarity = (
        np.dot(a, b)
        /
        (
            np.linalg.norm(a)
            * np.linalg.norm(b)
        )
    )

    return {
        "key": "semantic_similarity",
        "score": float(similarity)
    }

In [8]:
from langsmith.evaluation import evaluate

results = evaluate(
    target,
    data="Telecom-RAG-dataset",
    evaluators=[
        semantic_match
    ],
    experiment_prefix="telecom-rag-eval"
)

print(results)

View the evaluation results for experiment: 'telecom-rag-eval-3f0e1eb4' at:
https://smith.langchain.com/o/75acb5a5-eac1-484d-ba76-47a6fbeb21ac/datasets/9f34eabe-337d-4ed4-bea9-f8df7cbfea94/compare?selectedSessions=7149038f-4ee6-4988-b481-65e613fe250c




35it [06:54, 11.86s/it]

<ExperimentResults telecom-rag-eval-3f0e1eb4>
